# Demo search: от dense search до реранкинга
Выберите Python kernel проекта (`.venv`) и выполните **Run All**. Qdrant и Ollama должны быть запущены; Qwen3 0.6B уже установлена. Первый запуск может скачивать модели Hugging Face.

Ноутбук создаёт hybrid-коллекцию только при её отсутствии; существующую не удаляет и не обновляет. Все способы используют один индекс и один корпус. Финальный ответ LLM не генерируется: сравниваем поиск.

Чтобы открыть из терминала в корне проекта: `uv run jupyter lab demo-search.ipynb`.


In [1]:
from pathlib import Path
import sys
import time
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "llamaindex" / "advanced_rag_llamaindex_demo.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Откройте ноутбук из папки проекта или notebooks внутри неё.")
sys.path.insert(0, str(ROOT / "llamaindex"))
import advanced_rag_llamaindex_demo as demo
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.core.schema import NodeWithScore
from llama_index.core.postprocessor import SentenceTransformerRerank

MODEL = demo.REWRITE_MODEL
TOP_K, TOP_N = 8, 3
COLLECTION = demo.HYBRID_COLLECTION
# Настройки отображения: None — полный текст чанка; число — лимит символов.
FRAGMENT_MAX_CHARS = None
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
rows = []
print("Корень:", ROOT)
print("Исходные документы:", ROOT / "data" / "knowledge_base")
print("Python:", sys.executable)
if not demo.check_qdrant():
    raise RuntimeError("Запустите: docker compose up -d qdrant")
if not demo.check_ollama(MODEL):
    raise RuntimeError(f"Запустите Ollama и выполните: ollama pull {MODEL}")


Корень: /Users/newuser/Documents/repo/rag-engineering-workshop
Исходные документы: /Users/newuser/Documents/repo/rag-engineering-workshop/data/knowledge_base
Python: /Users/newuser/Documents/repo/rag-engineering-workshop/.venv/bin/python
Проверка Qdrant... OK, коллекций: 1
Версия Qdrant: 1.16.2
Проверка Ollama... OK
Проверка модели hf.co/Qwen/Qwen3-0.6B-GGUF:Q8_0... найдена


## 1. Подготовка одного общего индекса
36 Markdown-файлов лежат в `data/knowledge_base`. В Qdrant хранятся чанки, их текст, метаданные, dense BGE-M3 и sparse BM25. Существующий индекс может содержать более старую версию корпуса.


Исходные документы: `data/knowledge_base/` в корне проекта (абсолютный путь выводится выше). Для этого примера откройте `data/knowledge_base/IT_Zero_Trust_2026.md`. Полный фрагмент в таблице — это текст найденного чанка; исходный Markdown-файл может содержать несколько чанков.

Настройка `FRAGMENT_MAX_CHARS = None` показывает весь чанк. Для сокращённого вывода задайте, например, `450` и повторите Run All.


In [2]:
demo.configure_settings(MODEL, enable_tracing=False)
if demo.collection_exists(COLLECTION):
    index = demo.load_hybrid_index(collection_name=COLLECTION, model=MODEL, enable_tracing=False)
    print("Используем существующую коллекцию:", COLLECTION)
else:
    documents = demo.load_documents()
    print("Создаём индекс; документов:", len(documents))
    storage = StorageContext.from_defaults(vector_store=demo.hybrid_vector_store(COLLECTION))
    index = VectorStoreIndex.from_documents(documents, storage_context=storage, show_progress=True)
client = demo.QdrantClient(url=demo.QDRANT_URL)
info = client.get_collection(COLLECTION)
print("Точек (чанков) в коллекции:", info.points_count)
client.close()

# Загружаем cross-encoder один раз. Загрузка моделей не включается в время поиска.
t0 = time.perf_counter()
reranker = SentenceTransformerRerank(model=demo.RERANK_MODEL, top_n=TOP_N,
                                    device=demo.select_device(), keep_retrieval_score=False)
print(f"Инициализация reranker: {time.perf_counter() - t0:.1f} сек.")


Embedding model: BAAI/bge-m3 on mps
LLM via LlamaIndex: hf.co/Qwen/Qwen3-0.6B-GGUF:Q8_0 at http://localhost:11434
LLM request timeout: 600s


Embedding model: BAAI/bge-m3 on mps
LLM via LlamaIndex: hf.co/Qwen/Qwen3-0.6B-GGUF:Q8_0 at http://localhost:11434
LLM request timeout: 600s


Используем существующую коллекцию: robotex_docs_llamaindex_hybrid
Точек (чанков) в коллекции: 38


Инициализация reranker: 2.6 сек.


In [3]:
def fragment_text(node):
    text = node.node.get_content(metadata_mode="none")
    return text if FRAGMENT_MAX_CHARS is None else text[:FRAGMENT_MAX_CHARS]

def show_results(label, nodes, target, elapsed_ms, group, query):
    top = nodes[:TOP_N]
    rank = demo.rank_of_source(top, target)
    # Документная оценка: повторные чанки одного источника считаем один раз.
    relevant_sources = {target}
    retrieved_sources = {demo.source_filename(node) for node in top}
    found_relevant = relevant_sources & retrieved_sources
    precision = len(found_relevant) / len(retrieved_sources) if retrieved_sources else 0.0
    recall = len(found_relevant) / len(relevant_sources)
    table = pd.DataFrame([
        {"Ранг": r, "Источник": demo.source_filename(n), "Оценка": n.score,
         "Целевой источник": demo.source_filename(n) == target,
         "Фрагмент": fragment_text(n)}
        for r, n in enumerate(top, 1)
    ])
    print(label, "\nЗапрос:", query)
    display(table.style.set_properties(subset=["Фрагмент"],
        **{"white-space": "pre-wrap", "text-align": "left", "overflow-wrap": "anywhere"}))
    rows.append({"Задача": group, "Способ": label,
                 "Top-3 источники": " → ".join(demo.source_filename(n) for n in top),
                 "Целевой источник": target, "Ранг в top-3": rank,
                 "Hit@3": int(rank is not None), "MRR@3": 1 / rank if rank else 0,
                 "Precision@3": precision, "Recall@3": recall,
                 "Время, мс": round(elapsed_ms, 1)})
    return nodes

def search(label, query, target, group, *, hybrid=False, year=None, rerank=False, extra_ms=0):
    started = time.perf_counter()
    nodes = demo.retrieve_nodes(query=query, index=index, top_k=TOP_K, year=year, hybrid=hybrid)
    if rerank:
        demo.print_lecture_nodes("Кандидаты hybrid до реранкинга", nodes)
        nodes = reranker.postprocess_nodes(
            [NodeWithScore(node=n.node, score=n.score) for n in nodes], query_str=query)
    elapsed = (time.perf_counter() - started) * 1000 + extra_ms
    return show_results(label, nodes, target, elapsed, group, query)


## 2. VPN: последовательно добавляем меры
**Вопрос:** можно ли использовать VPN ежедневно, а не только при аварии?

**Разметка:** ожидаемый источник `IT_Zero_Trust_2026.md`. В нём VPN разрешён только для аварийного доступа. Инструкции установки VPN тематически похожи, но не дают нужной политики. Каждый этап ниже показывает top-3; retrieval собирает до восьми кандидатов.


In [4]:
QUESTION = demo.LECTURE_RERANK_QUESTION
TARGET = demo.LECTURE_RERANK_TARGET
GROUP = "VPN: допустимость ежедневного использования"
baseline = search("1. Dense без фильтра", QUESTION, TARGET, GROUP)


1. Dense без фильтра 
Запрос: Можно ли пользоваться VPN для обычной ежедневной работы, а не только при аварии?


,Ранг,Источник,Оценка,Целевой источник,Фрагмент
0,1,Doc_Filler_7.md,0.558556,False,"**Руководство по использованию VPN-клиента в компании РобоТех** 1. **Цель и назначение** Инструкция предназначена для описания процедуры установки и использования Virtual Private Network (VPN) клиента, обеспечивающего защищенное соединение с внешней сетью компании РобоТех. 2. **Установка VPN-клиента** Инсталляция VPN-клиента может быть выполнена следующими способами: - Скачивание и установка утилиты по ссылке, предоставленной администратором сети. - Установка VPN-клиента при помощи управляющей системы (Windows Update или App Store). 3. **Настройка VPN-клиента** После установки необходимо выполнить настройку VPN-клиента: - Зарегистрироваться в компании РобоТех, если это не сделано ранее. - Выбрать сервер расположенный в сети компании РобоТех. - Установить безопасное соединение и автоматическое подключение к серверу. 4. **Использование VPN-клиента** После успешной настройки VPN-клиент будет автоматически подключаться при включении компьютера и позволять работникам компании РобоТех безопасно обмениваться данными с внешней сетью. 5. **Контактная информация** Вопросы к настройке VPN-клиента можно задавать администратору сети компании РобоТех по электронной почте [admin@roboTech.com](mailto:admin@roboTech.com)."
1,2,Doc_Filler_19.md,0.553605,False,"# Внутреннее Руководство: Использование VPN-Клиента в РобоТехе 1. **Введение** В рамках обеспечения безопасности и эффективности работы, РобоТех предоставляет всем сотрудникам доступ к VPN-клиенту. Это руководство описывает основные требования и процедуры по использованию VPN-клиента для подключения к внутренней сети компании. 2. **Требования** - Для подключения к VPN-сети необходимо установленный на вашем компьютере или мобильном устройстве VPN-клиент, который предоставляется компанией. - Обязательно соблюдайте стандарты безопасности и политику конфиденциальности при работе с внутренней сетью. 3. **Процедура подключения** - Запустите VPN-клиент, который найден по умолчанию на вашем компьютере или мобильном устройстве. - Выберите сервер VPN в списке, предоставленном компанией. - Введите необходимые учетные данные (полученные от администратора) для аутентификации. - Проверьте успешное подключение к VPN-сети, используя команды ping или traceroute. 4. **Завершение работы** - Для завершения работы с VPN-клиентом нажмите соответствующую кнопку «Отключить» или выберите опцию ""Disconnect"" в меню приложения. - Контрольный лог аутентификации будет автоматически записан администраторами сети для дальнейшей проверки безопасности."
2,3,Doc_Filler_5.md,0.473398,False,"Инструкция №123456789-РТК ============================== **Область применения:** Внутренняя сфера **Дата выдачи:** 01.01.2023 **Содержание:** Список разрешенного программного обеспечения компании РобоТех **Цель документа:** Регулирование использования ПО в организации, обеспечение безопасности и эффективной работы. Всем сотрудникам компании РобоТех! В связи с необходимостью регулирования использования программного обеспечения в организации, мы приводим следующий список разрешенных ПО. Использование других программ без соответствующего разрешения запрещено и может быть наказано по внутренним правилам. 1. **Microsoft Office 365** – пакет офисных приложений для работы с текстом, таблицами, презентациями и графическими элементами. 2. **Adobe Creative Cloud** – пакет приложений для создания различной дизайнерской продукции: фото-, видео- и веб-проектов. 3. **Slack** – платформа для внутренней коммуникации, обмена сообщениями и файлами между сотрудниками компании. 4. **Google Workspace** – альтернативная платформа для работы с текстом, таблицами, презентациями и графическими элементами. При использовании указанного ПО необходимо следовать правилам безопасности, в том числе применять пароли сложной комплексности, обновлять системы регулярно и использовать защищенные каналы передачи данных. Если вы столкнетесь с необходимостью использования другого ПО или если 

### Добавляем фильтр `year=2026`
Фильтр ограничивает кандидатов метаданными. Год части документов определяется загрузчиком автоматически; это не полноценная проверка актуальности документа.


In [5]:
filtered = search("2. Dense + year=2026", QUESTION, TARGET, GROUP, year=2026)


2. Dense + year=2026 
Запрос: Можно ли пользоваться VPN для обычной ежедневной работы, а не только при аварии?


,Ранг,Источник,Оценка,Целевой источник,Фрагмент
0,1,Doc_Filler_7.md,0.558556,False,"**Руководство по использованию VPN-клиента в компании РобоТех** 1. **Цель и назначение** Инструкция предназначена для описания процедуры установки и использования Virtual Private Network (VPN) клиента, обеспечивающего защищенное соединение с внешней сетью компании РобоТех. 2. **Установка VPN-клиента** Инсталляция VPN-клиента может быть выполнена следующими способами: - Скачивание и установка утилиты по ссылке, предоставленной администратором сети. - Установка VPN-клиента при помощи управляющей системы (Windows Update или App Store). 3. **Настройка VPN-клиента** После установки необходимо выполнить настройку VPN-клиента: - Зарегистрироваться в компании РобоТех, если это не сделано ранее. - Выбрать сервер расположенный в сети компании РобоТех. - Установить безопасное соединение и автоматическое подключение к серверу. 4. **Использование VPN-клиента** После успешной настройки VPN-клиент будет автоматически подключаться при включении компьютера и позволять работникам компании РобоТех безопасно обмениваться данными с внешней сетью. 5. **Контактная информация** Вопросы к настройке VPN-клиента можно задавать администратору сети компании РобоТех по электронной почте [admin@roboTech.com](mailto:admin@roboTech.com)."
1,2,Doc_Filler_19.md,0.553605,False,"# Внутреннее Руководство: Использование VPN-Клиента в РобоТехе 1. **Введение** В рамках обеспечения безопасности и эффективности работы, РобоТех предоставляет всем сотрудникам доступ к VPN-клиенту. Это руководство описывает основные требования и процедуры по использованию VPN-клиента для подключения к внутренней сети компании. 2. **Требования** - Для подключения к VPN-сети необходимо установленный на вашем компьютере или мобильном устройстве VPN-клиент, который предоставляется компанией. - Обязательно соблюдайте стандарты безопасности и политику конфиденциальности при работе с внутренней сетью. 3. **Процедура подключения** - Запустите VPN-клиент, который найден по умолчанию на вашем компьютере или мобильном устройстве. - Выберите сервер VPN в списке, предоставленном компанией. - Введите необходимые учетные данные (полученные от администратора) для аутентификации. - Проверьте успешное подключение к VPN-сети, используя команды ping или traceroute. 4. **Завершение работы** - Для завершения работы с VPN-клиентом нажмите соответствующую кнопку «Отключить» или выберите опцию ""Disconnect"" в меню приложения. - Контрольный лог аутентификации будет автоматически записан администраторами сети для дальнейшей проверки безопасности."
2,3,Doc_Filler_5.md,0.473398,False,"Инструкция №123456789-РТК ============================== **Область применения:** Внутренняя сфера **Дата выдачи:** 01.01.2023 **Содержание:** Список разрешенного программного обеспечения компании РобоТех **Цель документа:** Регулирование использования ПО в организации, обеспечение безопасности и эффективной работы. Всем сотрудникам компании РобоТех! В связи с необходимостью регулирования использования программного обеспечения в организации, мы приводим следующий список разрешенных ПО. Использование других программ без соответствующего разрешения запрещено и может быть наказано по внутренним правилам. 1. **Microsoft Office 365** – пакет офисных приложений для работы с текстом, таблицами, презентациями и графическими элементами. 2. **Adobe Creative Cloud** – пакет приложений для создания различной дизайнерской продукции: фото-, видео- и веб-проектов. 3. **Slack** – платформа для внутренней коммуникации, обмена сообщениями и файлами между сотрудниками компании. 4. **Google Workspace** – альтернативная платформа для работы с текстом, таблицами, презентациями и графическими элементами. При использовании указанного ПО необходимо следовать правилам безопасности, в том числе применять пароли сложной комплексности, обновлять системы регулярно и использовать защищенные каналы передачи данных. Если вы столкнетесь с необходимостью использования другого ПО или если 

### Добавляем hybrid search
Dense отвечает за семантику, BM25 — за слова и обозначения. Объединение сигналов не гарантирует улучшение.


In [6]:
hybrid = search("3. Hybrid + year=2026", QUESTION, TARGET, GROUP, hybrid=True, year=2026)


3. Hybrid + year=2026 
Запрос: Можно ли пользоваться VPN для обычной ежедневной работы, а не только при аварии?


,Ранг,Источник,Оценка,Целевой источник,Фрагмент
0,1,Doc_Filler_7.md,1.000000,False,"**Руководство по использованию VPN-клиента в компании РобоТех** 1. **Цель и назначение** Инструкция предназначена для описания процедуры установки и использования Virtual Private Network (VPN) клиента, обеспечивающего защищенное соединение с внешней сетью компании РобоТех. 2. **Установка VPN-клиента** Инсталляция VPN-клиента может быть выполнена следующими способами: - Скачивание и установка утилиты по ссылке, предоставленной администратором сети. - Установка VPN-клиента при помощи управляющей системы (Windows Update или App Store). 3. **Настройка VPN-клиента** После установки необходимо выполнить настройку VPN-клиента: - Зарегистрироваться в компании РобоТех, если это не сделано ранее. - Выбрать сервер расположенный в сети компании РобоТех. - Установить безопасное соединение и автоматическое подключение к серверу. 4. **Использование VPN-клиента** После успешной настройки VPN-клиент будет автоматически подключаться при включении компьютера и позволять работникам компании РобоТех безопасно обмениваться данными с внешней сетью. 5. **Контактная информация** Вопросы к настройке VPN-клиента можно задавать администратору сети компании РобоТех по электронной почте [admin@roboTech.com](mailto:admin@roboTech.com)."
1,2,Doc_Filler_19.md,0.776903,False,"# Внутреннее Руководство: Использование VPN-Клиента в РобоТехе 1. **Введение** В рамках обеспечения безопасности и эффективности работы, РобоТех предоставляет всем сотрудникам доступ к VPN-клиенту. Это руководство описывает основные требования и процедуры по использованию VPN-клиента для подключения к внутренней сети компании. 2. **Требования** - Для подключения к VPN-сети необходимо установленный на вашем компьютере или мобильном устройстве VPN-клиент, который предоставляется компанией. - Обязательно соблюдайте стандарты безопасности и политику конфиденциальности при работе с внутренней сетью. 3. **Процедура подключения** - Запустите VPN-клиент, который найден по умолчанию на вашем компьютере или мобильном устройстве. - Выберите сервер VPN в списке, предоставленном компанией. - Введите необходимые учетные данные (полученные от администратора) для аутентификации. - Проверьте успешное подключение к VPN-сети, используя команды ping или traceroute. 4. **Завершение работы** - Для завершения работы с VPN-клиентом нажмите соответствующую кнопку «Отключить» или выберите опцию ""Disconnect"" в меню приложения. - Контрольный лог аутентификации будет автоматически записан администраторами сети для дальнейшей проверки безопасности."
2,3,Doc_Filler_14.md,0.309179,False,"**Регламент использования кофемашины на кухне** 1. **Сохранность и гигиена** Для обеспечения эффективного работы кофемашины необходимо соблюдать следующие меры: - При использовании кофемашины избегайте попадания жидкостей, в особенности капель масла, на корпус аппарата. - Ежедневно после завершения работы процеживайте кофейную машину и удаляйте остатки беспомощных кофеина. - При необходимости смазывайте внешнюю часть кофемашины специальным продуктом, предотвращающим скопление наносящих урон жидкостей. 2. **Регулярное обслуживание** Корпоративная кофемашина является крупной инвестицией для компании 'РобоТех'. Поэтому необходимо регулярно проводить техническое обслуживание аппарата. Рекомендуется проводить этот процесс каждые 6 месяцев, а в случае нарушения работы кофемашины - как можно скорее. 3. **Использование питания** Кофемашина должна подключаться к электросети с постоянным током 230В, частотой 50Гц и иметь максимальный расход тока до 13А. Не рекомендуется использование питания с переменной напряжением или низкого качества, поскольку это может привести к несчастным случаям и повреждению кофемашины. 4. **Доступность** Кофемашина должна быть доступна для всех сотрудников компании 'РобоТех'. Поэтому необходимо устраивать её в простом и легком доступе, а также располагать приблизительно к точкам сбора коллективов. При занятости кофемашины рекомендуется

### Добавляем cross-encoder
Модель читает пару «вопрос + чанк», пересортировывает восемь кандидатов и оставляет три. Если нужного чанка нет среди кандидатов, reranker его не найдёт. Retrieval score и rerank score имеют разные шкалы.


In [7]:
reranked = search("4. Hybrid + фильтр + rerank", QUESTION, TARGET, GROUP,
                  hybrid=True, year=2026, rerank=True)



=== Кандидаты hybrid до реранкинга ===
#1 retrieval_score=1.0000 Doc_Filler_7.md
    **Руководство по использованию VPN-клиента в компании РобоТех** 1. **Цель и назначение** Инструкция предназначена для описания процедуры установки и использования Virtual Private Network (VPN) клиента, обеспечивающего за
#2 retrieval_score=0.7769 Doc_Filler_19.md
    # Внутреннее Руководство: Использование VPN-Клиента в РобоТехе 1. **Введение** В рамках обеспечения безопасности и эффективности работы, РобоТех предоставляет всем сотрудникам доступ к VPN-клиенту. Это руководство описыв
#3 retrieval_score=0.3092 Doc_Filler_14.md
    **Регламент использования кофемашины на кухне** 1. **Сохранность и гигиена** Для обеспечения эффективного работы кофемашины необходимо соблюдать следующие меры: - При использовании кофемашины избегайте попадания жидкосте
#4 retrieval_score=0.2748 Doc_Filler_18.md
    # Internal Document: Friday Casual Dress Code Guidelines Выпуск №: 1.0.1 Дата выхода: Март 1, 2023 Обновления 

4. Hybrid + фильтр + rerank 
Запрос: Можно ли пользоваться VPN для обычной ежедневной работы, а не только при аварии?


,Ранг,Источник,Оценка,Целевой источник,Фрагмент
0,1,IT_Zero_Trust_2026.md,-0.528834,True,"# Стандарт доступа Zero Trust 2026 С 2026 года все инженерные сервисы РобоТех подключаются через Zero Trust Access. Внутренний код стандарта: ZTA-17. Для доступа к Qdrant, Neo4j, GitHub Actions и окружениям customer-data требуется корпоративный SSO, аппаратный ключ FIDO2 и активная проверка устройства. VPN остается резервным каналом только для аварийного доступа. Постоянное использование VPN вместо Zero Trust запрещено. Иван Петров отвечает за внедрение ZTA-17 в команде Platform. Команда Security проводит ежеквартальный аудит доступа. Связанные системы: Qdrant, Neo4j, Ollama, GitHub Actions."
1,2,Doc_Filler_7.md,-1.879383,False,"**Руководство по использованию VPN-клиента в компании РобоТех** 1. **Цель и назначение** Инструкция предназначена для описания процедуры установки и использования Virtual Private Network (VPN) клиента, обеспечивающего защищенное соединение с внешней сетью компании РобоТех. 2. **Установка VPN-клиента** Инсталляция VPN-клиента может быть выполнена следующими способами: - Скачивание и установка утилиты по ссылке, предоставленной администратором сети. - Установка VPN-клиента при помощи управляющей системы (Windows Update или App Store). 3. **Настройка VPN-клиента** После установки необходимо выполнить настройку VPN-клиента: - Зарегистрироваться в компании РобоТех, если это не сделано ранее. - Выбрать сервер расположенный в сети компании РобоТех. - Установить безопасное соединение и автоматическое подключение к серверу. 4. **Использование VPN-клиента** После успешной настройки VPN-клиент будет автоматически подключаться при включении компьютера и позволять работникам компании РобоТех безопасно обмениваться данными с внешней сетью. 5. **Контактная информация** Вопросы к настройке VPN-клиента можно задавать администратору сети компании РобоТех по электронной почте [admin@roboTech.com](mailto:admin@roboTech.com)."
2,3,Doc_Filler_14.md,-4.483132,False,"**Регламент использования кофемашины на кухне** 1. **Сохранность и гигиена** Для обеспечения эффективного работы кофемашины необходимо соблюдать следующие меры: - При использовании кофемашины избегайте попадания жидкостей, в особенности капель масла, на корпус аппарата. - Ежедневно после завершения работы процеживайте кофейную машину и удаляйте остатки беспомощных кофеина. - При необходимости смазывайте внешнюю часть кофемашины специальным продуктом, предотвращающим скопление наносящих урон жидкостей. 2. **Регулярное обслуживание** Корпоративная кофемашина является крупной инвестицией для компании 'РобоТех'. Поэтому необходимо регулярно проводить техническое обслуживание аппарата. Рекомендуется проводить этот процесс каждые 6 месяцев, а в случае нарушения работы кофемашины - как можно скорее. 3. **Использование питания** Кофемашина должна подключаться к электросети с постоянным током 230В, частотой 50Гц и иметь максимальный расход тока до 13А. Не рекомендуется использование питания с переменной напряжением или низкого качества, поскольку это может привести к несчастным случаям и повреждению кофемашины. 4. **Доступность** Кофемашина должна быть доступна для всех сотрудников компании 'РобоТех'. Поэтому необходимо устраивать её в простом и легком доступе, а также располагать приблизительно к точкам сбора коллективов. При занятости кофемашины рекомендуется ожидать своего хода. Следование этим правилам поможет улучшить качество работы кофемашины и обеспечить максимальную удобство для её использования сотрудниками компании 'РобоТех'."


## 3. Контекстный вопрос: Query Rewriting
Отдельная задача: история содержит тему ZTA-17, а вопрос «А кто за это отвечает?» сам по себе её не называет. Сравниваем один и тот же исходный вопрос до и после уточнения через Qwen3; фильтр и retrieval не меняем. Затем проверяем добавление rerank.


In [8]:
HISTORY = "До этого мы обсуждали стандарт доступа ZTA-17 для инженерных сервисов."
SHORT_QUERY = "А кто за это отвечает?"
CONTEXT_GROUP = "ZTA-17: ответственный из истории"
print("История:", HISTORY)
raw = search("5. Hybrid + фильтр, без rewrite", SHORT_QUERY, TARGET, CONTEXT_GROUP,
             hybrid=True, year=2026)


История: До этого мы обсуждали стандарт доступа ZTA-17 для инженерных сервисов.
5. Hybrid + фильтр, без rewrite 
Запрос: А кто за это отвечает?


,Ранг,Источник,Оценка,Целевой источник,Фрагмент
0,1,Doc_Filler_23.md,0.665376,False,"**Регламент Код-Ревью (Code Review)** 1. **Цель и Назначение** Представляется Регламент Код-Ревью, целью которого является обеспечение качества кода путем систематического анализа изменений в коде программного обеспечения. Этот процесс помогает улучшить качество кода и снизить риск внесения ошибок, вызывающих технические неполадки или безопасности в работе приложения. 2. **Обязанности участников** * **Автор**: отвечает за написание кода и его первичную проверку. Они должны обеспечить, что код соответствует всем требованиям, сформулированным в документах проекта, и не содержит ошибки или неоптимальные решения. * **Ревьюер**: проверяет код на соответствие стандартам и правилам компании, а также на качество и эффективность решений. Рецензент может предложить изменения или улучшения для обеспечения высокого уровня качества. * **Руководитель**: отвечает за координацию процесса Код-Ревью и одобрение финальных изменений в коде перед его интеграцией в основную ветку. Они также могут предоставлять рекомендации участникам для улучшения качества кода. 3. **Порядок выполнения** * Код, подлежащий проверке, должен быть отправлен на Код-Ревью через специальную систему или в группу рецензентов для анализа и оценки. * Рецензент должен сформировать обновления и комментарии по коду, после чего они должны быть одобрены автором и руководителем. * После получения одобрения, изменения в коде могут быть интегрированы в основную ветку приложения. 4. **Требования к Код-Ревью** * Код должен соответствовать стандартам компании и документации проекта. * Все изменения должны быть подтверждены автором и одобрены руководителем. * Рецензент обязан предоставить конструктивные комментарии и рекомендации для улучшения качества кода. * При необходимости, может проводиться дополнительный анализ кода, если возникают вопросы или ошибки в процессе работы приложения."
1,2,Project_Red_Eye.md,0.500000,False,"# Проект ""Красный Глаз"" (Red Eye) ## Введение В соответствии с поручением руководства компании РобоТех, прилагается внутренний документ, основанный на переданных фактах о проекте ""Красный Глаз"". Данный проект имеет секретное классификацию и целью является разработка системы зрения для дронов-доставщиков. ## Тело документа ### Описание проекта Проект ""Красный Глаз"" включает в себя создание высокопроизводительной системы искусственного интеллекта (AI) для обеспечения дронам-доставщикам улучшенного зрения. Эта система будет основана на библиотеке PyTorch с использованием CUDA для ускорения вычислений. ### Ответственный сотрудник Ответственным за проект ""Красный Глаз"" является инженер Виктор Цой, который имеет опыт работы в области искусственного интеллекта и дронов. ## Заключение Проект ""Красный Глаз"" является важной частью дальнейшего развития компании РобоТех и может обеспечить нам конкурентное преимущество в рынке дронов-доставщиков. Все сотрудники должны быть ознакомлены с секретностью проекта, не разглашая информацию третьим лицам. ## Дополнительные рекомендации Для обеспечения успешного завершения проекта ""Красный Глаз"", следует: - Соблюдать высочайшую степень секретности при работе с документацией и исходными кодами. - Постоянно обновлять знания в области PyTorch, CUDA и искусственного интеллекта для улучшения системы зрения. - Обеспечивать адекватное тестирование разработанных алгоритмов перед их имплементацией на дронах."
2,3,Meeting_Atlas_Security_2026.md,0.500000,False,"# Встреча Atlas и Security по графовому RAG Дата: 2026-03-14. Участники: Иван Петров, Мария Соколова, Анна Ким, команда Security, команда AI Platform. Решения: - Для графового поиска используется Neo4j PropertyGraphIndex. - Для точных кодов политик используется hybrid search с sparse-сигналом BM25. - Для финального порядка источников используется Cross-Encoder Reranking. - Для вопросов про текущие HR-правила обязателен metadata filter year=2026 и category=HR. Иван Петров должен подготовить демонст

In [9]:
started = time.perf_counter()
rewritten = demo.rewrite_query(SHORT_QUERY, HISTORY, model=MODEL)
rewrite_ms = (time.perf_counter() - started) * 1000
print("Переписанный запрос:", rewritten)
rewritten_nodes = search("6. Rewrite + hybrid + фильтр", rewritten, TARGET, CONTEXT_GROUP,
                         hybrid=True, year=2026, extra_ms=rewrite_ms)


Переписанный запрос: Кто в истории отвечает за доступ к системе ZTA-17 для инженерных сервисов?


6. Rewrite + hybrid + фильтр 
Запрос: Кто в истории отвечает за доступ к системе ZTA-17 для инженерных сервисов?


,Ранг,Источник,Оценка,Целевой источник,Фрагмент
0,1,IT_Zero_Trust_2026.md,0.967713,True,"# Стандарт доступа Zero Trust 2026 С 2026 года все инженерные сервисы РобоТех подключаются через Zero Trust Access. Внутренний код стандарта: ZTA-17. Для доступа к Qdrant, Neo4j, GitHub Actions и окружениям customer-data требуется корпоративный SSO, аппаратный ключ FIDO2 и активная проверка устройства. VPN остается резервным каналом только для аварийного доступа. Постоянное использование VPN вместо Zero Trust запрещено. Иван Петров отвечает за внедрение ZTA-17 в команде Platform. Команда Security проводит ежеквартальный аудит доступа. Связанные системы: Qdrant, Neo4j, Ollama, GitHub Actions."
1,2,Project_Atlas_Graph_2026.md,0.669703,False,"# Проект Atlas: графовая навигация по корпоративным знаниям Проект Atlas строит Graph RAG для корпоративных документов РобоТех. Цель проекта: связать документы, сотрудников, команды, политики и технические системы в одном property graph. Мария Соколова является владельцем бизнес-требований проекта Atlas. Иван Петров является техническим лидом проекта Atlas. Анна Ким отвечает за бюджет и закупки. Проект Atlas использует LlamaIndex PropertyGraphIndex для извлечения сущностей и отношений из документов. Граф хранится в Neo4j. Векторные чанки и hybrid search хранятся в Qdrant. Команда AI Platform отвечает за RAG pipeline. Команда Security отвечает за требования ZTA-17. Команда PeopleOps отвечает за политику HR-REMOTE-2026. Ключевые связи: - Мария Соколова owns HR-REMOTE-2026. - Иван Петров leads Project Atlas. - Project Atlas uses Neo4j. - Project Atlas uses Qdrant. - Project Atlas follows ZTA-17."
2,3,Project_Red_Eye.md,0.178569,False,"# Проект ""Красный Глаз"" (Red Eye) ## Введение В соответствии с поручением руководства компании РобоТех, прилагается внутренний документ, основанный на переданных фактах о проекте ""Красный Глаз"". Данный проект имеет секретное классификацию и целью является разработка системы зрения для дронов-доставщиков. ## Тело документа ### Описание проекта Проект ""Красный Глаз"" включает в себя создание высокопроизводительной системы искусственного интеллекта (AI) для обеспечения дронам-доставщикам улучшенного зрения. Эта система будет основана на библиотеке PyTorch с использованием CUDA для ускорения вычислений. ### Ответственный сотрудник Ответственным за проект ""Красный Глаз"" является инженер Виктор Цой, который имеет опыт работы в области искусственного интеллекта и дронов. ## Заключение Проект ""Красный Глаз"" является важной частью дальнейшего развития компании РобоТех и может обеспечить нам конкурентное преимущество в рынке дронов-доставщиков. Все сотрудники должны быть ознакомлены с секретностью проекта, не разглашая информацию третьим лицам. ## Дополнительные рекомендации Для обеспечения успешного завершения проекта ""Красный Глаз"", следует: - Соблюдать высочайшую степень секретности при работе с документацией и исходными кодами. - Постоянно обновлять знания в области PyTorch, CUDA и искусственного интеллекта для улучшения системы зрения. - Обеспечивать адекватное тестирование разработанных алгоритмов перед их имплементацией на дронах."


In [10]:
rewritten_reranked = search("7. Rewrite + hybrid + фильтр + rerank", rewritten, TARGET,
                            CONTEXT_GROUP, hybrid=True, year=2026, rerank=True,
                            extra_ms=rewrite_ms)



=== Кандидаты hybrid до реранкинга ===
#1 retrieval_score=0.9677 IT_Zero_Trust_2026.md
    # Стандарт доступа Zero Trust 2026 С 2026 года все инженерные сервисы РобоТех подключаются через Zero Trust Access. Внутренний код стандарта: ZTA-17. Для доступа к Qdrant, Neo4j, GitHub Actions и окружениям customer-data
#2 retrieval_score=0.6697 Project_Atlas_Graph_2026.md
    # Проект Atlas: графовая навигация по корпоративным знаниям Проект Atlas строит Graph RAG для корпоративных документов РобоТех. Цель проекта: связать документы, сотрудников, команды, политики и технические системы в одно
#3 retrieval_score=0.1786 Project_Red_Eye.md
    # Проект "Красный Глаз" (Red Eye) ## Введение В соответствии с поручением руководства компании РобоТех, прилагается внутренний документ, основанный на переданных фактах о проекте "Красный Глаз". Данный проект имеет секре
#4 retrieval_score=0.1294 Meeting_Atlas_Security_2026.md
    # Встреча Atlas и Security по графовому RAG Дата: 2026-03-14. Участники: Иван

7. Rewrite + hybrid + фильтр + rerank 
Запрос: Кто в истории отвечает за доступ к системе ZTA-17 для инженерных сервисов?


,Ранг,Источник,Оценка,Целевой источник,Фрагмент
0,1,IT_Zero_Trust_2026.md,5.714246,True,"# Стандарт доступа Zero Trust 2026 С 2026 года все инженерные сервисы РобоТех подключаются через Zero Trust Access. Внутренний код стандарта: ZTA-17. Для доступа к Qdrant, Neo4j, GitHub Actions и окружениям customer-data требуется корпоративный SSO, аппаратный ключ FIDO2 и активная проверка устройства. VPN остается резервным каналом только для аварийного доступа. Постоянное использование VPN вместо Zero Trust запрещено. Иван Петров отвечает за внедрение ZTA-17 в команде Platform. Команда Security проводит ежеквартальный аудит доступа. Связанные системы: Qdrant, Neo4j, Ollama, GitHub Actions."
1,2,Project_Atlas_Graph_2026.md,1.811361,False,"# Проект Atlas: графовая навигация по корпоративным знаниям Проект Atlas строит Graph RAG для корпоративных документов РобоТех. Цель проекта: связать документы, сотрудников, команды, политики и технические системы в одном property graph. Мария Соколова является владельцем бизнес-требований проекта Atlas. Иван Петров является техническим лидом проекта Atlas. Анна Ким отвечает за бюджет и закупки. Проект Atlas использует LlamaIndex PropertyGraphIndex для извлечения сущностей и отношений из документов. Граф хранится в Neo4j. Векторные чанки и hybrid search хранятся в Qdrant. Команда AI Platform отвечает за RAG pipeline. Команда Security отвечает за требования ZTA-17. Команда PeopleOps отвечает за политику HR-REMOTE-2026. Ключевые связи: - Мария Соколова owns HR-REMOTE-2026. - Иван Петров leads Project Atlas. - Project Atlas uses Neo4j. - Project Atlas uses Qdrant. - Project Atlas follows ZTA-17."
2,3,Doc_Filler_17.md,-5.916631,False,"# Внутренний регламент: Разрешенное программное обеспечение для использования в компании РобоТех 1. **Общие положения** В соответствии с политикой безопасности и эффективной работы в компании РобоТех, определены требования к программному обеспечению, которое разрешено для использования сотрудниками. 2. **Обязанности сотрудников** Каждый сотрудник обязан использовать только программное обеспечение, которое включено в список, одобренный ИТ-департаментом компании РобоТех. 3. **Список разрешенного программного обеспечения** Следующие приложения и системы являются разрешенными для использования сотрудниками компании РобоТех: - Microsoft Office 365 (Word, Excel, PowerPoint, Outlook) - Google Workspace (Docs, Sheets, Slides, Gmail) - Slack - Zoom - Adobe Creative Cloud (Photoshop, Illustrator, InDesign, Acrobat Pro) - Jira - Confluence - Trello 4. **Дополнительные требования** Кроме указанного программного обеспечения, сотрудники могут использовать дополнительное ПО при получении соответствующего разрешения от ИТ-департамента. Необходимо поддерживать обновления и используйте лицензионные версии программного обеспечения для соблюдения законов о авторских правах. 5. **Наказание за нарушения** Несоблюдение требований данного регламента может привести к дисциплинарным действиям, включая увольнение сотрудника."


## 4. Итоговая таблица качества поиска
- **Hit@3:** 1, если ожидаемый документ попал в первые три чанка; иначе 0.
- **MRR@3:** 1 / ранг первого чанка ожидаемого документа в top-3; иначе 0. Выше — лучше.
- **Precision@3:** число найденных релевантных документов / число уникальных документов среди первых трёх чанков. Повторные чанки одного файла считаем один раз.
- **Recall@3:** число найденных релевантных документов / число всех размеченных релевантных документов для вопроса.
- Здесь релевантный набор для каждого вопроса состоит из одного ожидаемого документа `IT_Zero_Trust_2026.md`. Поэтому Recall@3 равен Hit@3 (0 или 1). Это оценка относительно данной разметки, а не утверждение, что других релевантных документов нет.
- Ранги относятся к чанкам; один документ может повторяться.
- Время включает retrieval и применённые меры; для rewrite включает измеренное время одного переписывания. Загрузка индекса и инициализация моделей исключены. Это одиночные замеры без прогрева, а не бенчмарк.

Это диагностика двух вопросов по одному размеченному источнику на вопрос, а не оценка всей RAG-системы. Для вычисления метрик документы вне ожидаемого набора считаются нерелевантными; расширьте разметку для более полной оценки. Сравнивайте способы внутри одной задачи; улучшение не гарантируется.


In [11]:
comparison = pd.DataFrame(rows)
display(comparison.style.format({"MRR@3": "{:.3f}", "Precision@3": "{:.3f}", "Recall@3": "{:.3f}", "Время, мс": "{:.1f}"}, na_rep="—"))
comparison.to_csv(ROOT / "demo-search-comparison.csv", index=False)
print("Таблица сохранена в demo-search-comparison.csv")
print("Проверяем ответ в исходном документе:")
print((ROOT / "data" / "knowledge_base" / TARGET).read_text(encoding="utf-8"))


,Задача,Способ,Top-3 источники,Целевой источник,Ранг в top-3,Hit@3,MRR@3,Precision@3,Recall@3,"Время, мс"
0,VPN: допустимость ежедневного использования,1. Dense без фильтра,Doc_Filler_7.md → Doc_Filler_19.md → Doc_Filler_5.md,IT_Zero_Trust_2026.md,—,0,0.000,0.000,0.000,614.0
1,VPN: допустимость ежедневного использования,2. Dense + year=2026,Doc_Filler_7.md → Doc_Filler_19.md → Doc_Filler_5.md,IT_Zero_Trust_2026.md,—,0,0.000,0.000,0.000,59.5
2,VPN: допустимость ежедневного использования,3. Hybrid + year=2026,Doc_Filler_7.md → Doc_Filler_19.md → Doc_Filler_14.md,IT_Zero_Trust_2026.md,—,0,0.000,0.000,0.000,47.9
3,VPN: допустимость ежедневного использования,4. Hybrid + фильтр + rerank,IT_Zero_Trust_2026.md → Doc_Filler_7.md → Doc_Filler_14.md,IT_Zero_Trust_2026.md,1.000000,1,1.000,0.333,1.000,392.9
4,ZTA-17: ответственный из истории,"5. Hybrid + фильтр, без rewrite",Doc_Filler_23.md → Project_Red_Eye.md → Meeting_Atlas_Security_2026.md,IT_Zero_Trust_2026.md,—,0,0.000,0.000,0.000,88.6
5,ZTA-17: ответственный из истории,6. Rewrite + hybrid + фильтр,IT_Zero_Trust_2026.md → Project_Atlas_Graph_2026.md → Project_Red_Eye.md,IT_Zero_Trust_2026.md,1.000000,1,1.000,0.333,1.000,13681.2
6,ZTA-17: ответственный из истории,7. Rewrite + hybrid + фильтр + rerank,IT_Zero_Trust_2026.md → Project_Atlas_Graph_2026.md → Doc_Filler_17.md,IT_Zero_Trust_2026.md,1.000000,1,1.000,0.333,1.000,11794.5


Таблица сохранена в demo-search-comparison.csv
Проверяем ответ в исходном документе:
---
year: 2026
category: IT_Security
department: Security
doc_type: standard
access_group: employees
graph_ready: true
---
# Стандарт доступа Zero Trust 2026

С 2026 года все инженерные сервисы РобоТех подключаются через Zero Trust Access. Внутренний код стандарта: ZTA-17.

Для доступа к Qdrant, Neo4j, GitHub Actions и окружениям customer-data требуется корпоративный SSO, аппаратный ключ FIDO2 и активная проверка устройства.

VPN остается резервным каналом только для аварийного доступа. Постоянное использование VPN вместо Zero Trust запрещено.

Иван Петров отвечает за внедрение ZTA-17 в команде Platform. Команда Security проводит ежеквартальный аудит доступа.

Связанные системы: Qdrant, Neo4j, Ollama, GitHub Actions.


